# Rules of Prompt Caching

Caching isn't automatic — you place **cache breakpoints** on specific blocks. Claude caches everything **up to and including** the breakpoint; a follow-up request reuses that work only if the content up to the breakpoint is **byte-identical**.

> Rules/mechanics lesson (no live calls). The runnable demo is next: **Prompt caching in action**.

## Cache breakpoints

- Work is **not** cached automatically — you must add a breakpoint.
- Everything **before and including** the breakpoint gets cached.
- The cache is used on follow-ups **only if the content up to the breakpoint is identical** — even adding "please" invalidates it.

You add a breakpoint with the **longhand** block form (the shorthand string form has nowhere to attach it):

```python
# shorthand — no place for cache_control
{"type": "text", "text": "..."}

# longhand — can carry a breakpoint
{
    "type": "text",
    "text": "<large, stable content>",
    "cache_control": {"type": "ephemeral"}
}
```

## What you can cache, and the ordering

Breakpoints aren't limited to text — they work on **system prompts, tool definitions, images, and tool_use / tool_result blocks**. System prompts and tool definitions are the best candidates: they're large and rarely change.

Claude assembles the request in a fixed order, and the cache prefix follows it:

```
1. tools     (tool definitions)
2. system    (system prompt)
3. messages  (conversation)
```

A change at any level invalidates that level **and everything after it**. So cache the most-stable things (tools, system) and let the volatile tail (the latest user message) fall after the last breakpoint.

**Cross-message:** a breakpoint in a later message caches all *earlier* messages too — handy for caching a whole conversation prefix.

**Up to 4 breakpoints** per request. Typical use: one after tools, one after system, one partway through history.

## Illustrative placement

Caching a big system prompt + tools, leaving the user's question uncached after the last breakpoint:

```python
client.messages.create(
    model=model,
    max_tokens=1000,
    tools=[
        { ...large tool schema..., "cache_control": {"type": "ephemeral"} },  # breakpoint after tools
    ],
    system=[
        {"type": "text", "text": "<long, stable system prompt>",
         "cache_control": {"type": "ephemeral"}},                            # breakpoint after system
    ],
    messages=[
        {"role": "user", "content": "A short question that changes every call"},  # not cached
    ],
)
```

## Minimum content length — ⚠️ course is out of date here

Content must exceed a **minimum token count** (summed across everything up to the breakpoint, not per-block) to be cacheable. The course says **1024**, but the real threshold is **model-dependent** — and notably **higher on Haiku**:

| Model | Minimum tokens to cache |
|-------|-------------------------|
| Fable 5 / Mythos 5 | 512 |
| Sonnet 5 / Sonnet 4.x / Opus 4.8 / Opus 4.1 | 1,024 |
| Opus 4.5–4.7 | ~2,048–4,096 |
| Haiku 3.5 | 2,048 |
| **Haiku 4.5** | **4,096** |

**This matters for the next lesson:** we run on Haiku 4.5, so a cached prefix under **4,096 tokens silently won't cache** (`cache_creation_input_tokens` stays 0). The course's "duplicate a short message ~500×" trick assumes 1,024 — we'll need a bigger prefix to see caching kick in.

## Other current-API facts (verified against docs)

- **Syntax:** `cache_control: {"type": "ephemeral"}` — correct and current; caching is **GA** (no beta header).
- **TTL:** default **5 minutes** (refreshes free on each hit); request the **1-hour** tier with `{"type": "ephemeral", "ttl": "1h"}`. When mixing, longer TTLs must appear *before* shorter ones.
- **Pricing:** cache **write** ≈ 1.25× input (5m) / 2× (1h); cache **read** ≈ 0.1× input.
- **Usage response:** `cache_creation_input_tokens` (written), `cache_read_input_tokens` (reused at 0.1×), `input_tokens` (everything after the last breakpoint — not cached).

Next: **Prompt caching in action** — place a real breakpoint and watch the usage fields prove the cache is working.